In [0]:
from pyspark.sql.functions import format_number
from pyspark.sql import functions as F

In [0]:
import logging
from datetime import datetime

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load from Silver Delta table
logger.info("Loading silver table...")

df_bronze = spark.read.table("workspace.default.ppr_bronze")

logger.info(f"Loaded {df_bronze.count()} records from silver table")
df_bronze.printSchema()

In [0]:
logger.info("Starting Bronze to Silver transformation...")

df_silver = df_bronze \
    .withColumn("date_of_sale", F.to_date("date_of_sale", "dd/MM/yyyy")) \
    .withColumn("price", F.regexp_replace("price_raw", "[€,]", "").cast("double")) \
    .withColumn("year", F.year("date_of_sale")) \
    .withColumn("month", F.month("date_of_sale")) \
    .drop("price_raw") \
    .filter(F.col("price").isNotNull()) \
    .filter(F.col("county").isNotNull())

print(f"Silver records: {df_silver.count()}")
df_silver.printSchema()

# Save Silver table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.ppr_silver")

logger.info(f"Silver records saved: {df_silver.count()}")
logger.info("Silver table saved: workspace.default.ppr_silver")